In [ ]:
!pip install -U transformers datasets accelerate evaluate

In [ ]:
!pip install -U transformers accelerate -q

In [ ]:
!pip install librosa soundfile audiomentations -q

In [ ]:
!pip install -U peft -q

In [ ]:
!pip install -U torchao -q

In [2]:
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

BASE = Path("/kaggle/working/ser-india")
RAW  = BASE / "data/raw"
PROC = BASE / "data/processed"

for p in [RAW/"shemo", RAW/"iiith", PROC]:
    p.mkdir(parents=True, exist_ok=True)

print("Folders ready")

Folders ready


In [3]:
SHEMO = RAW / "shemo_repo"
SHEMO.mkdir(parents=True, exist_ok=True)

print("Shemo folder created at:", SHEMO)

Shemo folder created at: /kaggle/working/ser-india/data/raw/shemo_repo


In [4]:
import shutil
from pathlib import Path

src_paths = [
    Path("/kaggle/input/datasets/apoorv007/shemorepo/female"),
    Path("/kaggle/input/datasets/apoorv007/shemorepo/male")
]

# Copy folders
for src in src_paths:
    dst = SHEMO / src.name
    shutil.copytree(src, dst, dirs_exist_ok=True)

print("Files copied to:", SHEMO)

Files copied to: /kaggle/working/ser-india/data/raw/shemo_repo


In [5]:
shemo_repo = RAW / "shemo_repo"
wav_files = list(shemo_repo.rglob("*.wav"))
print(f"Total WAV files found: {len(wav_files)}")
print("\nSample filenames:")
for f in wav_files[:8]:
    print(f"  {f.name}  →  parent: {f.parent.name}")

Total WAV files found: 3000

Sample filenames:
  M43A11.wav  →  parent: male
  M40N31.wav  →  parent: male
  M04S03.wav  →  parent: male
  M27N44.wav  →  parent: male
  M26A27.wav  →  parent: male
  M46N09.wav  →  parent: male
  M04N51.wav  →  parent: male
  M12N06.wav  →  parent: male


In [6]:
import pandas as pd
from pathlib import Path

data = []

for file in wav_files:
    filename = file.name
    
    # Extract emotion code (example: N, H, A, S, F, D)
    emotion_code = filename[3]  # check this based on your dataset
    
    data.append({
        "path": str(file),
        "emotion": emotion_code
    })

df = pd.DataFrame(data)
df.head()

,path,emotion
0,/kaggle/working/ser-india/data/raw/shemo_repo/...,A
1,/kaggle/working/ser-india/data/raw/shemo_repo/...,N
2,/kaggle/working/ser-india/data/raw/shemo_repo/...,S
3,/kaggle/working/ser-india/data/raw/shemo_repo/...,N
4,/kaggle/working/ser-india/data/raw/shemo_repo/...,A


Business DOES NOT care about:

happiness
disgust
Map to business classes:

In [7]:
mapping = {
    'A': 'frustrated',   # anger
    'N': 'calm',         # neutral
    'H': 'calm',         # happiness
    'W': 'disengaged'    # surprise (best approximation)
}

df["label"] = df["emotion"].map(mapping)
df.to_csv('shemo.csv', index=False)
df.head()

,path,emotion,label
0,/kaggle/working/ser-india/data/raw/shemo_repo/...,A,frustrated
1,/kaggle/working/ser-india/data/raw/shemo_repo/...,N,calm
2,/kaggle/working/ser-india/data/raw/shemo_repo/...,S,NaN
3,/kaggle/working/ser-india/data/raw/shemo_repo/...,N,calm
4,/kaggle/working/ser-india/data/raw/shemo_repo/...,A,frustrated


In [8]:
df = pd.read_csv("/kaggle/working/shemo.csv")   # ← change to your actual CSV path
print(df.head())
print(df.columns.tolist())
print(df['label'].value_counts())       # ← change 'label' to your actual column name if different
print(f"Total samples: {len(df)}")

                                                path emotion       label
0  /kaggle/working/ser-india/data/raw/shemo_repo/...       A  frustrated
1  /kaggle/working/ser-india/data/raw/shemo_repo/...       N        calm
2  /kaggle/working/ser-india/data/raw/shemo_repo/...       S         NaN
3  /kaggle/working/ser-india/data/raw/shemo_repo/...       N        calm
4  /kaggle/working/ser-india/data/raw/shemo_repo/...       A  frustrated
['path', 'emotion', 'label']
label
calm          1229
frustrated    1059
disengaged     225
Name: count, dtype: int64
Total samples: 3000


In [9]:
LABEL2ID = {
    "calm":        0,
    "frustrated":  1,
    "disengaged":  2,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

print(LABEL2ID)
print(f"Number of classes: {NUM_LABELS}")

{'calm': 0, 'frustrated': 1, 'disengaged': 2}
Number of classes: 3


Phone audio augmentation

In [10]:
def simulate_phone_audio(audio: np.ndarray, sr: int = 16000) -> np.ndarray:
    # Codec compression: downsample to 8kHz then back
    degraded = librosa.resample(audio, orig_sr=sr, target_sr=8000)
    degraded = librosa.resample(degraded, orig_sr=8000, target_sr=sr)

    # Gaussian noise at ~12dB SNR
    signal_power = np.mean(degraded ** 2) + 1e-9
    noise_power  = signal_power / (10 ** (12 / 10))
    noise        = np.random.normal(0, np.sqrt(noise_power), len(degraded))
    degraded     = degraded + noise

    # Clip to safe range
    degraded = np.clip(degraded, -1.0, 1.0)
    return degraded.astype(np.float32)

print("Augmentation function ready")

Augmentation function ready


In [11]:
import torch
import torchaudio
from torch.utils.data import Dataset

TARGET_SR      = 16000
MAX_SAMPLES    = TARGET_SR * 6   # 6 seconds max per clip

class ShemoDataset(Dataset):
    def __init__(self, df, processor, augment=False):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.augment   = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        file_path = row["path"]
        label     = LABEL2ID[row["label"]]

        # Load audio
        waveform, sr = torchaudio.load(file_path)

        # Mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample to 16kHz
        if sr != TARGET_SR:
            waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)

        audio = waveform.squeeze().numpy()

        # Truncate or pad to 6 seconds
        if len(audio) > MAX_SAMPLES:
            audio = audio[:MAX_SAMPLES]
        else:
            audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))

        # Phone degradation on training data only
        if self.augment:
            audio = simulate_phone_audio(audio)

        inputs = self.processor(
            audio,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding=True,
        )

        return {
            "input_values": inputs.input_values.squeeze(),
            "labels":       torch.tensor(label, dtype=torch.long),
        }

print("Dataset class ready")

Dataset class ready


In [12]:
from peft import LoraConfig, get_peft_model
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    Trainer,
    TrainingArguments,
)

MODEL_CKPT = "facebook/wav2vec2-base"
processor = Wav2Vec2Processor.from_pretrained(MODEL_CKPT)

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=NUM_LABELS,
    label2id=LABEL2ID,
    id2label=ID2LABEL,
    ignore_mismatched_sizes=True,
)

# Quick check before wrapping — confirm attention module names exist in this transformers version
for name, _ in model.named_modules():
    if "q_proj" in name or "v_proj" in name:
        print(name)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_hid.bias             | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


wav2vec2.encoder.layers.0.attention.v_proj
wav2vec2.encoder.layers.0.attention.q_proj
wav2vec2.encoder.layers.1.attention.v_proj
wav2vec2.encoder.layers.1.attention.q_proj
wav2vec2.encoder.layers.2.attention.v_proj
wav2vec2.encoder.layers.2.attention.q_proj
wav2vec2.encoder.layers.3.attention.v_proj
wav2vec2.encoder.layers.3.attention.q_proj
wav2vec2.encoder.layers.4.attention.v_proj
wav2vec2.encoder.layers.4.attention.q_proj
wav2vec2.encoder.layers.5.attention.v_proj
wav2vec2.encoder.layers.5.attention.q_proj
wav2vec2.encoder.layers.6.attention.v_proj
wav2vec2.encoder.layers.6.attention.q_proj
wav2vec2.encoder.layers.7.attention.v_proj
wav2vec2.encoder.layers.7.attention.q_proj
wav2vec2.encoder.layers.8.attention.v_proj
wav2vec2.encoder.layers.8.attention.q_proj
wav2vec2.encoder.layers.9.attention.v_proj
wav2vec2.encoder.layers.9.attention.q_proj
wav2vec2.encoder.layers.10.attention.v_proj
wav2vec2.encoder.layers.10.attention.q_proj
wav2vec2.encoder.layers.11.attention.v_proj
wav2vec2

In [13]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    modules_to_save=["classifier", "projector"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

[transformers] Wav2Vec2ForSequenceClassification does not expose input embeddings. Gradients cannot flow back to the token embeddings when using adapters or gradient checkpointing. Override `get_input_embeddings` to fully support those features, or set `_input_embed_layer` to the attribute name that holds the embeddings.


trainable params: 492,547 || all params: 95,061,894 || trainable%: 0.5181


In [14]:
# Aggressive cleaning
df = df.dropna(subset=["path", "label"])
df = df[df["label"].isin(LABEL2ID.keys())]
df = df[df["path"].notna()]
df = df[df["path"] != ""]
df["label"] = df["label"].astype(str).str.strip()
df = df[df["label"].isin(LABEL2ID.keys())].reset_index(drop=True)

# Verify no NaN remains
print(f"NaN in label: {df['label'].isna().sum()}")
print(f"NaN in path:  {df['path'].isna().sum()}")
print(f"Clean samples: {len(df)}")
print(df["label"].value_counts())

NaN in label: 0
NaN in path:  0
Clean samples: 2513
label
calm          1229
frustrated    1059
disengaged     225
Name: count, dtype: int64


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=42,
)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}")
print("\nTrain distribution:")
print(train_df["label"].value_counts())

train_dataset = ShemoDataset(train_df, processor, augment=True)
val_dataset   = ShemoDataset(val_df,   processor, augment=False)

# Class weights to handle disengaged (225 samples) being underrepresented
classes = np.array(list(LABEL2ID.keys()))
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"].values,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).cuda()
print(f"\nClass weights: {dict(zip(LABEL2ID.keys(), class_weights.round(3)))}")
print("Datasets ready")

Train: 2136  |  Val: 377

Train distribution:
label
calm          1045
frustrated     900
disengaged     191
Name: count, dtype: int64

Class weights: {'calm': np.float64(0.681), 'frustrated': np.float64(0.791), 'disengaged': np.float64(3.728)}
Datasets ready


In [16]:
from dataclasses import dataclass
from typing import List, Dict
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    Trainer,
    TrainingArguments,
)
import evaluate

@dataclass
class DataCollatorForAudio:
    processor: Wav2Vec2Processor

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_values = [f["input_values"] for f in features]
        labels       = torch.tensor([f["labels"] for f in features], dtype=torch.long)

        batch = self.processor.pad(
            {"input_values": input_values},
            padding=True,
            return_tensors="pt",
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorForAudio(processor=processor)

f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=preds, references=labels, average="weighted")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("Collator, metric, WeightedTrainer ready")

Collator, metric, WeightedTrainer ready


In [19]:
OUTPUT_DIR = "/kaggle/working/wav2vec2-shemo-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    learning_rate=3e-5,
    warmup_steps=0.1,       # was warmup_ratio=0.1 — v5 merged them, float < 1 = ratio
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    dataloader_num_workers=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=20,
    report_to="none",
    fp16=True,
    gradient_accumulation_steps=2,
)
print("Training arguments ready")

Training arguments ready


In [20]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,2.202133,1.091147,0.250218
2,2.140370,1.078833,0.363288
3,2.063806,1.066013,0.425357
4,1.968631,1.052820,0.512969
5,1.891112,1.041735,0.371033
6,1.737209,1.036280,0.333444
7,1.651175,1.032443,0.303841
8,1.601675,1.029586,0.299757
9,1.548380,1.028766,0.294020
10,1.507290,1.028337,0.294020


TrainOutput(global_step=340, training_loss=1.815666675567627, metrics={'train_runtime': 1295.9986, 'train_samples_per_second': 16.481, 'train_steps_per_second': 0.262, 'total_flos': 1.16958070416384e+18, 'train_loss': 1.815666675567627, 'epoch': 10.0})

In [21]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,F1
1.507290,1.052820,10,0.512969


{'eval_loss': 1.0528203248977661, 'eval_f1': 0.512969458267264}

In [22]:
OUTPUT_DIR = "/kaggle/working/wav2vec2-shemo-lora"
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print("Model saved")

Model saved


In [23]:
!zip -r my_model_lora.zip /kaggle/working/wav2vec2-shemo-lora

  adding: kaggle/working/wav2vec2-shemo-lora/ (stored 0%)
  adding: kaggle/working/wav2vec2-shemo-lora/README.md (deflated 66%)
  adding: kaggle/working/wav2vec2-shemo-lora/checkpoint-170/ (stored 0%)
  adding: kaggle/working/wav2vec2-shemo-lora/checkpoint-170/README.md (deflated 66%)
  adding: kaggle/working/wav2vec2-shemo-lora/checkpoint-170/scheduler.pt (deflated 61%)
  adding: kaggle/working/wav2vec2-shemo-lora/checkpoint-170/training_args.bin (deflated 53%)
  adding: kaggle/working/wav2vec2-shemo-lora/checkpoint-170/scaler.pt (deflated 64%)
  adding: kaggle/working/wav2vec2-shemo-lora/checkpoint-170/optimizer.pt (deflated 7%)
  adding: kaggle/working/wav2vec2-shemo-lora/checkpoint-170/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/wav2vec2-shemo-lora/checkpoint-170/rng_state.pth (deflated 27%)
  adding: kaggle/working/wav2vec2-shemo-lora/checkpoint-170/trainer_state.json (deflated 69%)
  adding: kaggle/working/wav2vec2-shemo-lora/checkpoint-170/adapter_config.jso

This Fine Tuning can be done better using IIITH Hindi Audio dataset combining with the Shemo one... For now we'll Go with the dataset we have now(SHEMO)

In [24]:
import time
import numpy as np
import torch

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Use a handful of real val clips, not synthetic noise — latency on your actual data distribution
sample_batch = [val_dataset[i] for i in range(20)]

def benchmark_inference(model, samples, n_warmup=3, n_runs=20):
    # Warmup — first few calls are always slower (CUDA kernel compilation, cache misses)
    with torch.no_grad():
        for i in range(n_warmup):
            inputs = samples[i]["input_values"].unsqueeze(0).to(device)
            _ = model(inputs)

    torch.cuda.synchronize() if device == "cuda" else None

    latencies = []
    with torch.no_grad():
        for i in range(n_runs):
            inputs = samples[i % len(samples)]["input_values"].unsqueeze(0).to(device)
            start = time.perf_counter()
            _ = model(inputs)
            torch.cuda.synchronize() if device == "cuda" else None
            end = time.perf_counter()
            latencies.append((end - start) * 1000)  # ms

    return {
        "mean_ms": np.mean(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "std_ms": np.std(latencies),
    }

lora_latency = benchmark_inference(model, sample_batch)
print("LoRA model latency:", lora_latency)

LoRA model latency: {'mean_ms': np.float64(35.49146405000556), 'p50_ms': np.float64(35.39169650002805), 'p95_ms': np.float64(37.10887134973291), 'std_ms': np.float64(1.1192788098928763)}


In [25]:
import torch

# Dynamic int8 quantization — quantizes Linear layers, works on CPU
# (PyTorch's dynamic quantization backend doesn't support CUDA directly)
model_cpu = model.to("cpu")
model_cpu.eval()

quantized_model = torch.quantization.quantize_dynamic(
    model_cpu,
    {torch.nn.Linear},   # quantize all Linear layers to int8
    dtype=torch.qint8,
)

print("Quantization done")

Quantization done
